In [1]:
import bempp_cl.api
import numpy as np
import pytest


In [2]:
# Physical parameters
B_0 = 1.0  # Incident field magnitude

mu_0 = 1.0  # Normalized permeability

# Create geometry (sphere)
grid = bempp_cl.api.shapes.sphere(h=0.2, r=1.0)
print(f"Grid: {grid.number_of_elements} elements")




Grid: 846 elements


In [3]:
# grid.plot()

In [4]:
div_space  = bempp_cl.api.function_space(grid, "RWG", 0)
curl_space = bempp_cl.api.function_space(grid, "SNC", 0)
p1_space   = bempp_cl.api.function_space(grid, "P", 1) 
print(f"Div space dimension: {div_space.global_dof_count}")
print(f"Curl space dimension: {curl_space.global_dof_count}")
print(f"P1 space dimension: {p1_space.global_dof_count}")


Div space dimension: 1269
Curl space dimension: 1269
P1 space dimension: 425


In [5]:
@bempp_cl.api.real_callable
def incident_field_tangential(x, n, domain_index, result):
    B_inc = np.array([0.0 * x[2], 0.0 * x[2], B_0 + 0.0 * x[2]])
    result[:] = np.cross(B_inc, n)


In [6]:
# Create grid function for RHS
rhs_fun = bempp_cl.api.GridFunction(div_space, fun=incident_field_tangential, dual_space=curl_space)
print(f"RHS grid function created")



RHS grid function created


In [7]:
# Single layer boundary operator
V_op = bempp_cl.api.operators.boundary.maxwell.single_layer(div_space, div_space, curl_space)

# Single layer boundary operator
K_op = bempp_cl.api.operators.boundary.maxwell.double_layer(div_space, div_space, curl_space)


In [8]:
# Maxwell identity operator (for proper inner product)
identity_p1 = bempp_cl.api.operators.boundary.sparse.identity(
    p1_space, p1_space, p1_space
)


identity = bempp_cl.api.operators.boundary.sparse.identity(
    div_space, div_space, curl_space
)


vector_grad = bempp_cl.api.operators.boundary.sparse._vector_grad_product(
    p1_space, div_space, curl_space
)


In [9]:
identity_p1.weak_form()

<425x425 SparseDiscreteBoundaryOperator with dtype=float64>

In [10]:
# Solve the linear system
# from bempp_cl.api.linalg import lu, gmres

# Solution
# j_solution = lu(V_op, rhs)

In [11]:
# help(gmres)

In [12]:
from bempp_cl.api.linalg import lu, gmres

rhs = (0.5 * identity + K_op) * rhs_fun 

j_solution, j_info, j_count = gmres(V_op, rhs, tol=1e-6, return_iteration_count=True)

print(f"Number of GMRES iterations: {j_count}")

/home/ignacio/Documents/Github/bempp-cl/bempp_cl/api/assembly/discrete_boundary_operator.py:619: SparseEfficiencyWarning: splu converted its input to CSC format
  solver = solver_interface(actual_mat)


Number of GMRES iterations: 66


In [13]:
jmin = np.min(j_solution.coefficients)
jmax = np.max(j_solution.coefficients)

print(f"Minimum value: {jmin}\nMaximum value: {jmax}")

Minimum value: -121.72296127573178
Maximum value: 40.83266364245565


## Block operator

We assemble the block operator
\begin{equation*}
A_h = \begin{pmatrix} 
V_h & B^T_h \\ 
B_h^T & 0
\end{pmatrix},
\end{equation*}
where

- $V_h$ is the Galerkin boundary element matrix associated to the (vector) single layer operator.
- $B_h^T$ is the Galerkin matrix for the gradient of P1 functions, tested with RT.
- $B_h$ is the weak form of the surface divergence operator.

Matrix $A_h$ then corresponds to a discretization of the (vector) single layer operator and a Lagrange multiplier for the divergence-free condition on the fields.
  


In [14]:
# Block operator
from bempp_cl.api.assembly.blocked_operator import BlockedDiscreteOperator


blocks = [[None, None], [None, None]]

B = vector_grad.weak_form()

blocks[0][0] = V_op.weak_form()
blocks[0][1] = B
blocks[1][0] = -B.T
blocks[1][1] = 0*identity_p1.weak_form()


A = BlockedDiscreteOperator(np.array(blocks))


In [15]:
rhs_p1 = np.zeros(p1_space.global_dof_count, dtype=float)

RHS = np.concatenate([rhs.projections(curl_space), rhs_p1])

In [16]:
RHS

array([ 0.00840407, -0.01278478,  0.00744408, ...,  0.        ,
        0.        ,  0.        ], shape=(1694,))

In [36]:

from scipy.sparse.linalg import gmres

it_count = 0

def count_iterations(x):
    global it_count
    it_count += 1

SOL, info = gmres(A, RHS, rtol = 1e-6, restart=1000, maxiter= 1, callback=count_iterations, callback_type='pr_norm')

print(f"Number of GMRES iterations: {it_count}")
print(f"GMRES converged to a solution: {info == 0}")

Number of GMRES iterations: 655
GMRES converged to a solution: True


In [32]:
info

0

In [33]:
sol_bem    = SOL[:div_space.global_dof_count]
sol_lambda = SOL[div_space.global_dof_count:]

In [34]:
sol_bem

array([ -1.34408002,  -3.28362961,   1.73653659, ...,  -2.69321792,
         0.40418166, -19.12070857], shape=(1269,))

In [35]:
sol_lambda

array([-1.50800072e-02,  1.01653116e-01, -7.36326613e-03, -1.21050912e-02,
       -1.64138259e-02, -9.22897180e-03, -1.40859588e-02, -1.24739957e-02,
       -6.89663600e-03, -4.72912986e-03, -2.35864700e-03,  8.91480114e-03,
        9.41990387e-02,  6.48774340e-02,  4.86876932e-02,  2.68060852e-02,
        1.48302084e-02,  7.60957219e-03,  3.76208985e-03, -1.15337806e-05,
       -1.04059629e-02, -1.48508944e-02, -1.57517566e-02, -9.94544357e-03,
       -6.81311170e-03, -3.89964476e-03, -1.17865215e-02, -9.84514888e-03,
       -9.66555560e-03, -1.12038340e-02, -1.31736769e-02, -1.27480165e-02,
       -1.16533100e-02, -1.54031054e-02,  6.76760166e-02,  1.58349338e-02,
        1.64064306e-02,  1.85433353e-02,  1.78323173e-02,  1.78696436e-02,
       -1.26642228e-03, -1.68032469e-02, -9.47211490e-03, -6.89598645e-03,
       -1.14473313e-02, -1.66183469e-02, -1.72543340e-02, -1.22987903e-02,
       -1.05872163e-02, -9.66483353e-03, -7.97766922e-03, -7.20088918e-03,
       -8.15059451e-03, -

In [22]:
# slp_op.weak_form().to_dense()